# Demo 2 — Which features and split rule respect later-date prediction?

**Learning question:** At each Station A prediction issue, which candidate inputs are available by the feature cutoff, and which split represents evaluation on later target dates?

The input grain is one synthetic Station A daily prediction issue. The output grains are one row per candidate-feature decision and one row per stable split assignment. This Colab-first notebook runs equivalently in local Jupyter. Colab files are ephemeral, and edits opened from GitHub are not automatically saved back to GitHub. Assignment use of Colab remains conditional on the repository-save and Classroom 50 pilot. Use only the synthetic, non-identifying fixture; do not add private data or credentials. Restart the kernel and run every cell in order because stored output is not fresh-execution evidence.

In [ ]:
from importlib import metadata
from pathlib import Path
import platform
import subprocess
import sys

EXPECTED_PYTHON = "3.12.13"
EXPECTED_DISTRIBUTIONS = {
    "numpy": "2.0.2",
    "pandas": "3.0.3",
    "statsmodels": "0.14.6",
    "scikit-learn": "1.9.0",
    "matplotlib": "3.11.1",
}

def distribution_version(distribution_name):
    try:
        return metadata.version(distribution_name)
    except metadata.PackageNotFoundError:
        return None

mismatched = [
    f"{name}=={expected}"
    for name, expected in EXPECTED_DISTRIBUTIONS.items()
    if distribution_version(name) != expected
]
if mismatched:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *mismatched]
    )

actual_versions = {
    name: distribution_version(name) for name in EXPECTED_DISTRIBUTIONS
}
assert platform.python_version() == EXPECTED_PYTHON
assert actual_versions == EXPECTED_DISTRIBUTIONS

starting_directory = Path.cwd().resolve()
search_bases = (starting_directory, *starting_directory.parents)
demo_root = None
for search_base in search_bases:
    for candidate in (search_base, search_base / "10" / "demo"):
        if (candidate / "DEMO_GUIDE.md").is_file() and (
            candidate / ".python-version"
        ).is_file():
            demo_root = candidate
            break
    if demo_root is not None:
        break
DEMO_ROOT = demo_root if demo_root is not None else starting_directory

print(f"Python {platform.python_version()}")
for distribution_name, distribution_version_text in actual_versions.items():
    print(f"{distribution_name} {distribution_version_text}")
print(f"Demo root: {DEMO_ROOT}")

## Fix the prediction contract before choosing features

A **prediction timestamp** is when a prediction is issued. The **target** is the value to predict; its **target timestamp** says when that value occurs. The **prediction horizon** is the one-day gap between issue and target. A **feature** is an input value, and the **feature cutoff** is the latest time information may be used. **Information availability** asks whether every source needed for a candidate was known by that cutoff. The **primary metric** is the predeclared measure used to compare approaches; here it will be validation MAE.

Unit: one Station A daily prediction issue. Issue time: current day at 00:00 UTC. Target: next-day temperature in degrees C. Target time: issue time plus one day. Horizon: one day. Feature cutoff: the issue time.

In [ ]:
import hashlib
import io
import numpy as np
import pandas as pd

STATION_BYTES = b'row_id,prediction_timestamp,target_timestamp,day_number,current_temperature_c,previous_temperature_c,target_next_day_temperature_c\nstation-a-20260102,2026-01-02T00:00:00Z,2026-01-03T00:00:00Z,1,10.752852,10.400000,11.150020\nstation-a-20260103,2026-01-03T00:00:00Z,2026-01-04T00:00:00Z,2,11.150020,10.752852,12.284133\nstation-a-20260104,2026-01-04T00:00:00Z,2026-01-05T00:00:00Z,3,12.284133,11.150020,12.891635\nstation-a-20260105,2026-01-05T00:00:00Z,2026-01-06T00:00:00Z,4,12.891635,12.284133,12.500011\nstation-a-20260106,2026-01-06T00:00:00Z,2026-01-07T00:00:00Z,5,12.500011,12.891635,12.432889\nstation-a-20260107,2026-01-07T00:00:00Z,2026-01-08T00:00:00Z,6,12.432889,12.500011,12.810600\nstation-a-20260108,2026-01-08T00:00:00Z,2026-01-09T00:00:00Z,7,12.810600,12.432889,12.319227\nstation-a-20260109,2026-01-09T00:00:00Z,2026-01-10T00:00:00Z,8,12.319227,12.810600,11.265068\nstation-a-20260110,2026-01-10T00:00:00Z,2026-01-11T00:00:00Z,9,11.265068,12.319227,11.008799\nstation-a-20260111,2026-01-11T00:00:00Z,2026-01-12T00:00:00Z,10,11.008799,11.265068,11.042981\nstation-a-20260112,2026-01-12T00:00:00Z,2026-01-13T00:00:00Z,11,11.042981,11.008799,10.294535\nstation-a-20260113,2026-01-13T00:00:00Z,2026-01-14T00:00:00Z,12,10.294535,11.042981,9.694338\nstation-a-20260114,2026-01-14T00:00:00Z,2026-01-15T00:00:00Z,13,9.694338,10.294535,10.196415\nstation-a-20260115,2026-01-15T00:00:00Z,2026-01-16T00:00:00Z,14,10.196415,9.694338,10.705477\nstation-a-20260116,2026-01-16T00:00:00Z,2026-01-17T00:00:00Z,15,10.705477,10.196415,10.582814\nstation-a-20260117,2026-01-17T00:00:00Z,2026-01-18T00:00:00Z,16,10.582814,10.705477,11.069374\nstation-a-20260118,2026-01-18T00:00:00Z,2026-01-19T00:00:00Z,17,11.069374,10.582814,12.415247\nstation-a-20260119,2026-01-19T00:00:00Z,2026-01-20T00:00:00Z,18,12.415247,11.069374,13.203857\nstation-a-20260120,2026-01-20T00:00:00Z,2026-01-21T00:00:00Z,19,13.203857,12.415247,13.408874\nstation-a-20260121,2026-01-21T00:00:00Z,2026-01-22T00:00:00Z,20,13.408874,13.203857,14.297838\nstation-a-20260122,2026-01-22T00:00:00Z,2026-01-23T00:00:00Z,21,14.297838,13.408874,15.417233\nstation-a-20260123,2026-01-23T00:00:00Z,2026-01-24T00:00:00Z,22,15.417233,14.297838,15.482652\nstation-a-20260124,2026-01-24T00:00:00Z,2026-01-25T00:00:00Z,23,15.482652,15.417233,15.179048\nstation-a-20260125,2026-01-25T00:00:00Z,2026-01-26T00:00:00Z,24,15.179048,15.482652,15.559942\nstation-a-20260126,2026-01-26T00:00:00Z,2026-01-27T00:00:00Z,25,15.559942,15.179048,15.665661\nstation-a-20260127,2026-01-27T00:00:00Z,2026-01-28T00:00:00Z,26,15.665661,15.559942,14.738241\nstation-a-20260128,2026-01-28T00:00:00Z,2026-01-29T00:00:00Z,27,14.738241,15.665661,14.027121\nstation-a-20260129,2026-01-29T00:00:00Z,2026-01-30T00:00:00Z,28,14.027121,14.738241,14.098535\nstation-a-20260130,2026-01-30T00:00:00Z,2026-01-31T00:00:00Z,29,14.098535,14.027121,13.708819\nstation-a-20260131,2026-01-31T00:00:00Z,2026-02-01T00:00:00Z,30,13.708819,14.098535,12.768661\nstation-a-20260201,2026-02-01T00:00:00Z,2026-02-02T00:00:00Z,31,12.768661,13.708819,12.688712\nstation-a-20260202,2026-02-02T00:00:00Z,2026-02-03T00:00:00Z,32,12.688712,12.768661,13.310430\nstation-a-20260203,2026-02-03T00:00:00Z,2026-02-04T00:00:00Z,33,13.310430,12.688712,13.338624\nstation-a-20260204,2026-02-04T00:00:00Z,2026-02-05T00:00:00Z,34,13.338624,13.310430,13.290932\nstation-a-20260205,2026-02-05T00:00:00Z,2026-02-06T00:00:00Z,35,13.290932,13.338624,14.302447\nstation-a-20260206,2026-02-06T00:00:00Z,2026-02-07T00:00:00Z,36,14.302447,13.290932,15.487204\nstation-a-20260207,2026-02-07T00:00:00Z,2026-02-08T00:00:00Z,37,15.487204,14.302447,15.821827\nstation-a-20260208,2026-02-08T00:00:00Z,2026-02-09T00:00:00Z,38,15.821827,15.487204,16.311473\nstation-a-20260209,2026-02-09T00:00:00Z,2026-02-10T00:00:00Z,39,16.311473,15.821827,17.563960\nstation-a-20260210,2026-02-10T00:00:00Z,2026-02-11T00:00:00Z,40,17.563960,16.311473,18.266177\n'
STATION_SHA256 = "f95330b252c6e0f12026577602c69e21d01dbec232b5e523c6c41b0b62cf85a8"
TIMESTAMP_FORMAT = "%Y-%m-%dT%H:%M:%SZ"
fixture_path = DEMO_ROOT / "data" / "station_next_day.csv"
fixture_bytes = fixture_path.read_bytes() if fixture_path.is_file() else STATION_BYTES
assert len(fixture_bytes) == 3879
assert hashlib.sha256(fixture_bytes).hexdigest() == STATION_SHA256

station_data = pd.read_csv(
    io.BytesIO(fixture_bytes),
    dtype={
        "row_id": "string",
        "prediction_timestamp": "string",
        "target_timestamp": "string",
        "day_number": "int64",
        "current_temperature_c": "float64",
        "previous_temperature_c": "float64",
        "target_next_day_temperature_c": "float64",
    },
)
for timestamp_column in ["prediction_timestamp", "target_timestamp"]:
    station_data[timestamp_column] = pd.to_datetime(
        station_data[timestamp_column], format=TIMESTAMP_FORMAT, utc=True
    ).astype("datetime64[us, UTC]")
assert station_data.shape == (40, 7)
assert station_data["row_id"].is_unique
assert station_data.notna().all().all()
assert [str(dtype) for dtype in station_data.dtypes] == [
    "string", "datetime64[us, UTC]", "datetime64[us, UTC]",
    "int64", "float64", "float64", "float64"
]
assert station_data["prediction_timestamp"].is_monotonic_increasing
assert (
    station_data["target_timestamp"] - station_data["prediction_timestamp"]
    == pd.Timedelta(days=1)
).all()

OUTPUT_DIR = DEMO_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
demo2_output_names = ["availability_decisions.csv", "split_manifest.csv"]
for output_name in demo2_output_names:
    owned_path = OUTPUT_DIR / output_name
    if owned_path.exists():
        owned_path.unlink()

display(station_data.head())

## Audit availability before constructing candidates

**Target leakage** uses information revealed by the outcome. **Temporal leakage** uses information later than the feature cutoff. **Preprocessing leakage** lets held-out rows influence learned transformation state. **Test-set leakage** uses the final test partition for a feature, preprocessing, model, setting, or stopping decision.

Audit issue `station-a-20260125` at 2026-01-25 00:00 UTC. Before running the inventory, predict which two candidates require later information. The inventory will describe rejected candidates but will not compute them.

In [ ]:
prediction_cutoff = pd.Timestamp("2026-01-25T00:00:00Z").as_unit("us")
availability_decisions = pd.DataFrame({
    "candidate": pd.Series([
        "day_number",
        "current_temperature_c",
        "previous_temperature_c",
        "post_outcome_temperature_review_c",
        "full_dataset_scaled_current_temperature_c",
    ], dtype="string"),
    "latest_required_timestamp": pd.Series(pd.to_datetime([
        "2026-01-25T00:00:00Z",
        "2026-01-25T00:00:00Z",
        "2026-01-24T00:00:00Z",
        "2026-01-26T00:00:00Z",
        "2026-02-10T00:00:00Z",
    ], format=TIMESTAMP_FORMAT, utc=True), dtype="datetime64[us, UTC]"),
    "available_by_cutoff": pd.Series([True, True, True, False, False], dtype="bool"),
    "decision": pd.Series(["keep", "keep", "keep", "reject", "reject"], dtype="string"),
    "leakage_type": pd.Series([
        "none", "none", "none", "target/temporal", "preprocessing"
    ], dtype="string"),
})
assert (
    availability_decisions["available_by_cutoff"]
    == (availability_decisions["latest_required_timestamp"] <= prediction_cutoff)
).all()
assert availability_decisions["decision"].tolist() == [
    "keep", "keep", "keep", "reject", "reject"
]

availability_path = OUTPUT_DIR / "availability_decisions.csv"
availability_decisions.to_csv(
    availability_path,
    index=False,
    lineterminator="\n",
    date_format=TIMESTAMP_FORMAT,
)
assert len(availability_path.read_bytes()) == 416
assert hashlib.sha256(availability_path.read_bytes()).hexdigest() == (
    "e765b4426412525b06fcfad9717af158e786e7e0ff32c570341927192d6020f2"
)
display(availability_decisions)

## Define split roles before assigning rows

**Training** rows estimate model state. **Validation** rows compare development choices. **Test** rows are an untouched final check used only after choices are frozen. Rows are **exchangeable** for a split when their order is irrelevant to the intended sampling/deployment process. A **random seed** makes a random procedure reproducible. A **chronological split** assigns earlier and later time blocks from fixed cutoffs. A **split manifest** records each row's role stably.

First reproduce a two-stage random split for exchangeable teaching IDs. Before running it, remember that reproducibility alone does not make it appropriate for predicting later dates.

In [ ]:
from sklearn.model_selection import train_test_split

exchangeable_ids = np.arange(30, dtype="int64")
exchangeable_development, exchangeable_test = train_test_split(
    exchangeable_ids,
    test_size=0.20,
    random_state=217,
)
exchangeable_train, exchangeable_validation = train_test_split(
    exchangeable_development,
    test_size=0.25,
    random_state=217,
)
exchangeable_roles = {
    "train": sorted(exchangeable_train.tolist()),
    "validation": sorted(exchangeable_validation.tolist()),
    "test": sorted(exchangeable_test.tolist()),
}
assert exchangeable_roles == {
    "train": [1, 3, 6, 7, 9, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 22, 24, 28],
    "validation": [0, 8, 10, 25, 27, 29],
    "test": [2, 4, 5, 15, 23, 26],
}
exchangeable_sets = [set(values) for values in exchangeable_roles.values()]
assert [len(values) for values in exchangeable_sets] == [18, 6, 6]
assert set.union(*exchangeable_sets) == set(exchangeable_ids)
assert all(
    exchangeable_sets[left].isdisjoint(exchangeable_sets[right])
    for left in range(3) for right in range(left + 1, 3)
)
display(pd.DataFrame({
    "split": list(exchangeable_roles),
    "sorted_ids": [str(values) for values in exchangeable_roles.values()],
}))

In [ ]:
validation_start = pd.Timestamp("2026-01-25T00:00:00Z").as_unit("us")
test_start = pd.Timestamp("2026-02-01T00:00:00Z").as_unit("us")
temporal_roles = np.select(
    [
        station_data["target_timestamp"] < validation_start,
        station_data["target_timestamp"] < test_start,
    ],
    ["train", "validation"],
    default="test",
)
split_manifest = station_data[
    ["row_id", "prediction_timestamp", "target_timestamp"]
].copy()
split_manifest["split"] = pd.Series(temporal_roles, dtype="string")
assert split_manifest["row_id"].is_unique
assert split_manifest["split"].value_counts().to_dict() == {
    "train": 22, "test": 11, "validation": 7
}
role_id_sets = {
    role: set(split_manifest.loc[split_manifest["split"] == role, "row_id"])
    for role in ["train", "validation", "test"]
}
assert set.union(*role_id_sets.values()) == set(station_data["row_id"])
assert all(
    role_id_sets[left].isdisjoint(role_id_sets[right])
    for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]
)
role_times = {
    role: split_manifest.loc[split_manifest["split"] == role, "target_timestamp"]
    for role in ["train", "validation", "test"]
}
assert role_times["train"].min() == pd.Timestamp("2026-01-03T00:00:00Z").as_unit("us")
assert role_times["train"].max() == pd.Timestamp("2026-01-24T00:00:00Z").as_unit("us")
assert role_times["validation"].min() == validation_start
assert role_times["validation"].max() == pd.Timestamp("2026-01-31T00:00:00Z").as_unit("us")
assert role_times["test"].min() == test_start
assert role_times["test"].max() == pd.Timestamp("2026-02-11T00:00:00Z").as_unit("us")
assert role_times["train"].max() < role_times["validation"].min() < role_times["test"].min()

manifest_path = OUTPUT_DIR / "split_manifest.csv"
split_manifest.to_csv(
    manifest_path,
    index=False,
    lineterminator="\n",
    date_format=TIMESTAMP_FORMAT,
)
assert len(manifest_path.read_bytes()) == 2755
assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == (
    "92a1fb047929e318d0f0259634cfa454bad58076779dbc5058992a5b70e3d9d0"
)
display(split_manifest.groupby("split", sort=False).agg(
    rows=("row_id", "size"),
    first_target=("target_timestamp", "min"),
    last_target=("target_timestamp", "max"),
))

## Explain the decisions

The seeded split is reproducible, but later-date prediction is not exchangeable: a chronological split represents the intended future-facing handoff. `post_outcome_temperature_review_c` is rejected because it requires target-time information after the feature cutoff. `full_dataset_scaled_current_temperature_c` is rejected because its learned transformation state would include future validation/test feature rows. Neither rejected series was constructed.

Keep test untouched: do not use its results for a feature, preprocessing step, model, setting, or stopping choice.

In [ ]:
availability_readback = pd.read_csv(
    availability_path,
    dtype={
        "candidate": "string",
        "latest_required_timestamp": "string",
        "available_by_cutoff": "bool",
        "decision": "string",
        "leakage_type": "string",
    },
)
availability_readback["latest_required_timestamp"] = pd.to_datetime(
    availability_readback["latest_required_timestamp"],
    format=TIMESTAMP_FORMAT,
    utc=True,
).astype("datetime64[us, UTC]")
manifest_readback = pd.read_csv(
    manifest_path,
    dtype={
        "row_id": "string",
        "prediction_timestamp": "string",
        "target_timestamp": "string",
        "split": "string",
    },
)
for timestamp_column in ["prediction_timestamp", "target_timestamp"]:
    manifest_readback[timestamp_column] = pd.to_datetime(
        manifest_readback[timestamp_column], format=TIMESTAMP_FORMAT, utc=True
    ).astype("datetime64[us, UTC]")
assert [str(dtype) for dtype in availability_readback.dtypes] == [
    "string", "datetime64[us, UTC]", "bool", "string", "string"
]
assert [str(dtype) for dtype in manifest_readback.dtypes] == [
    "string", "datetime64[us, UTC]", "datetime64[us, UTC]", "string"
]
assert availability_readback.equals(availability_decisions)
assert manifest_readback.equals(split_manifest)
assert len(availability_path.read_bytes()) == 416
assert hashlib.sha256(availability_path.read_bytes()).hexdigest() == "e765b4426412525b06fcfad9717af158e786e7e0ff32c570341927192d6020f2"
assert len(manifest_path.read_bytes()) == 2755
assert hashlib.sha256(manifest_path.read_bytes()).hexdigest() == "92a1fb047929e318d0f0259634cfa454bad58076779dbc5058992a5b70e3d9d0"
assert set(path.name for path in OUTPUT_DIR.iterdir() if path.name in demo2_output_names) == set(demo2_output_names)
print("PASS: Demo 2 feature decisions and stable split manifest verified.")